# Sync vs Async

The script runs 3 fake "requests" — each one just sleeps for 2 seconds to simulate waiting on something (a network call, a database, etc).

**Sync:** the requests run one after another. Each one blocks until it finishes. Total time: ~6s.

**Async:** all 3 start at the same time. While each one is sleeping, the others aren't blocked — they're all waiting in parallel. Total time: ~2s.

The only real difference in the code is `async def`, `await`, and `asyncio.gather()`. The logic is identical — async just stops hogging the thread while it waits.


In [1]:
from IPython.display import display, Markdown

with open("01_sync_vs_async.py", encoding="utf-8") as f:
    code = f.read()

display(Markdown(f"```python\n{code}\n```"))


```python
import asyncio
import time


# ─── SYNCHRONOUS ────────────────────────────────────────────────────────────
# Each call blocks until it finishes — nothing else can run while it waits.

def fetch_sync(name):
    print(f"  [{name}] fetching...")
    time.sleep(2)
    print(f"  [{name}] done")
    return {"result": name}


def run_sync():
    start = time.time()
    fetch_sync("request-1")
    fetch_sync("request-2")
    fetch_sync("request-3")
    print(f"\nSync total: {time.time() - start:.1f}s\n")


# ─── ASYNCHRONOUS ───────────────────────────────────────────────────────────
# All three start immediately. While one is waiting, the others keep going.

async def fetch_async(name):
    print(f"  [{name}] fetching...")
    await asyncio.sleep(2)
    print(f"  [{name}] done")
    return {"result": name}


async def run_async():
    start = time.time()
    await asyncio.gather(
        fetch_async("request-1"),
        fetch_async("request-2"),
        fetch_async("request-3"),
    )
    print(f"\nAsync total: {time.time() - start:.1f}s\n")


# ─── ENTRY POINT ────────────────────────────────────────────────────────────

async def main():
    print("=== SYNC (blocks on each call) ===")
    run_sync()

    print("=== ASYNC (all run concurrently) ===")
    await run_async()


if __name__ == "__main__":
    asyncio.run(main())


# The key mental model: await means "I'm waiting, go do something
# else" — gather means "schedule all of these concurrently so they
# interleave at their await points instead of running one after another"

# async def — marks a function as one that can pause. That's it.

# await — the pause point. When Python hits await asyncio.sleep(2), 
# it pauses that function and goes to do other work. 

# asyncio — the Python library that manages all of this. 
# It runs an event loop — a scheduler that watches all paused functions 
# and wakes them up when they're ready.

# asyncio.gather() — hands multiple coroutines to the event loop at the 
# same time. Instead of starting one, waiting, then starting the next — 
# all three start immediately and the event loop juggles them while they wait.
```

In [2]:
!python 01_sync_vs_async.py


=== SYNC (blocks on each call) ===
  [request-1] fetching...
  [request-1] done
  [request-2] fetching...
  [request-2] done
  [request-3] fetching...
  [request-3] done

Sync total: 6.0s

=== ASYNC (all run concurrently) ===
  [request-1] fetching...
  [request-2] fetching...
  [request-3] fetching...
  [request-1] done
  [request-2] done
  [request-3] done

Async total: 2.0s



# Coroutine Basics

Calling an `async def` function does **not** run it — it returns a coroutine object. You have to `await` it to actually execute it and get the return value.

This is the single most important thing to understand about async: `async def` just marks the function, `await` is what runs it.

In [3]:
from IPython.display import display, Markdown

with open("02_coroutine_basics.py", encoding="utf-8") as f:
    code = f.read()

display(Markdown(f"```python\n{code}\n```"))

```python
import asyncio


# ─── REGULAR FUNCTION ───────────────────────────────────────────────────────
# Calling it runs it immediately and returns the value.

def greet():
    return "Hello"


# ─── COROUTINE FUNCTION ─────────────────────────────────────────────────────
# Adding `async` makes it a coroutine function.
# Calling it does NOT run it — it returns a coroutine object (a paused task).
# You need to await it to actually execute it.

async def greet_async():
    return "Hello from async"


async def main():
    # Regular function — runs immediately
    result = greet()
    print(result)                   # Hello

    # Coroutine function — calling without await gives you the object, not the value
    coro = greet_async()
    print(type(coro))               # <class 'coroutine'>
    coro.close()                    # discard it cleanly

    # await actually runs it and gives you the return value
    result = await greet_async()
    print(result)                   # Hello from async


if __name__ == "__main__":
    asyncio.run(main())

```

In [4]:
!python 02_coroutine_basics.py

Hello
<class 'coroutine'>
Hello from async


# Await in Action

Two patterns for when to use each:

**Sequential `await`:** step two depends on step one's result, so you wait for each in order. Total time is the sum.

**`asyncio.gather()`:** tasks are independent, so all start at once. Total time is the slowest task, not the sum.

In [5]:
from IPython.display import display, Markdown

with open("03_await_in_action.py", encoding="utf-8") as f:
    code = f.read()

display(Markdown(f"```python\n{code}\n```"))

```python
import asyncio
import time


# ─── SEQUENTIAL AWAIT ───────────────────────────────────────────────────────
# Each step waits for the previous one to finish before starting.
# Use this when step two depends on step one's result.

async def step_one():
    await asyncio.sleep(1)
    return "step one result"


async def step_two(input_data: str):
    await asyncio.sleep(0.5)
    return f"processed: {input_data}"


async def run_sequential():
    start = time.time()
    data = await step_one()         # wait for step one
    output = await step_two(data)   # then run step two with its result
    print(f"Sequential: {output!r} — took {time.time() - start:.1f}s")


# ─── CONCURRENT AWAIT ───────────────────────────────────────────────────────
# asyncio.gather() runs all coroutines at the same time.
# Total time = slowest task, not sum of all tasks.

async def fetch_tool(name: str, delay: float):
    await asyncio.sleep(delay)
    return f"{name} result"


async def run_concurrent():
    start = time.time()

    # These three run simultaneously — total ~1s, not 1+0.5+0.8=2.3s
    results = await asyncio.gather(
        fetch_tool("tool_call",    1.0),
        fetch_tool("resource_read", 0.5),
        fetch_tool("prompt_fetch",  0.8),
    )

    for r in results:
        print(f"  {r}")
    print(f"Concurrent: took {time.time() - start:.1f}s")


# ─── ENTRY POINT ────────────────────────────────────────────────────────────

async def main():
    print("=== Sequential (step two needs step one's output) ===")
    await run_sequential()

    print("\n=== Concurrent (independent tasks, run together) ===")
    await run_concurrent()


if __name__ == "__main__":
    asyncio.run(main())

```

In [6]:
!python 03_await_in_action.py

=== Sequential (step two needs step one's output) ===
Sequential: 'processed: step one result' — took 1.5s

=== Concurrent (independent tasks, run together) ===
  tool_call result
  resource_read result
  prompt_fetch result
Concurrent: took 1.0s


# Async With

`with` is a context manager — it runs setup on enter and teardown on exit, even if an error occurs. The classic example is `open()`: it opens the file and guarantees it closes.

`async with` does the same but when the setup or teardown itself requires async work — like a network handshake. In MCP, every client session is opened and closed this way.

In [7]:
from IPython.display import display, Markdown

with open("04_async_with.py", encoding="utf-8") as f:
    code = f.read()

display(Markdown(f"```python\n{code}\n```"))

```python
import asyncio
import os


# ─── WHAT IS a context manager? ──────────────────────────────────────────────
# A context manager handles setup and teardown around a block of code —
# guaranteeing cleanup even if an error occurs.
#
# Syntax: `with resource as name:`


# The classic example — open() is a context manager built into Python.
# It opens the file on enter and closes it on exit, no matter what.

with open("example.txt", "w") as f:
    f.write("hello")
# file is guaranteed closed here, even if an exception was raised inside
os.remove("example.txt")


# Without a context manager you have to manage cleanup manually:

f = open("example.txt", "w")
try:
    f.write("hello")
finally:
    f.close()   # easy to forget; and easy to miss if an early return appears
os.remove("example.txt")


# ─── HOW context managers work internally ─────────────────────────────────────
# A class with __enter__ and __exit__ is a context manager.
# __enter__ runs on entering the `with` block — return value binds to `as name`.
# __exit__  runs on leaving — always, even if the body raised an exception.

class DatabaseConnection:
    def __enter__(self):
        print("DB: connecting...")
        return self             # what gets bound to `as conn`

    def __exit__(self, exc_type, exc_val, exc_tb):
        print("DB: closing connection")
        if exc_type:
            print(f"DB: rolling back due to {exc_type.__name__}")
        return False            # False = let the exception propagate

with DatabaseConnection() as conn:
    print("DB: running query")
    # __exit__ is called here — connection always closed


# ─── WHAT IS async with? ──────────────────────────────────────────────────────
# `async with` does the same as `with` but when opening or closing the resource
# itself requires async work — like a network handshake or a graceful shutdown.
# Use __aenter__ and __aexit__ instead of __enter__ and __exit__.
#
# In MCP, every client session is opened and closed this way.


class MCPSession:
    async def __aenter__(self):
        print("[Session] Connecting...")
        await asyncio.sleep(0.1)        # simulate network handshake
        print("[Session] Connected.")
        return self

    async def __aexit__(self, *args):
        print("[Session] Closing...")
        await asyncio.sleep(0.05)       # simulate graceful shutdown
        print("[Session] Closed.")

    async def call_tool(self, name: str, args: dict) -> str:
        await asyncio.sleep(0.1)        # simulate round-trip
        return f"{name} returned: {args}"


async def main():
    # The session is guaranteed to close even if an exception is raised inside
    async with MCPSession() as session:
        result = await session.call_tool("weather", {"city": "Tokyo"})
        print(result)

    # Real MCP client code looks exactly like this:
    #
    # async with stdio_client(server_params) as (read, write):
    #     async with ClientSession(read, write) as session:
    #         await session.initialize()
    #         result = await session.call_tool("my_tool", {"input": "hello"})
    #
    # The nesting is intentional — two separate layers of cleanup:
    #   Outer: manages the raw byte stream (stdin/stdout pipe to the server)
    #   Inner: manages the MCP protocol session built on top of that stream
    # Both layers are guaranteed closed even if an exception occurs inside.


if __name__ == "__main__":
    asyncio.run(main())

```

In [8]:
!python 04_async_with.py

DB: connecting...
DB: running query
DB: closing connection
[Session] Connecting...
[Session] Connected.
weather returned: {'city': 'Tokyo'}
[Session] Closing...
[Session] Closed.


# Async Context Manager

`@asynccontextmanager` lets you write an async context manager as a function instead of a class. Everything before `yield` is setup, everything after is teardown. The `yield` value is what gets bound to `as name`.

This is the exact pattern MCP uses for server lifespans — wrapping startup and shutdown around the `yield`.

In [9]:
from IPython.display import display, Markdown

with open("04b_async_contextmanager.py", encoding="utf-8") as f:
    code = f.read()

display(Markdown(f"```python\n{code}\n```"))

```python
import asyncio
from collections.abc import AsyncIterator
from contextlib import asynccontextmanager


# ─── CREATING async context managers ────────────────────────────────────────
# 04_async_with.py taught you to USE async context managers (async with).
# 05_async_for.py  will teach you async generators    (async def + yield).
#
# This file sits between them: @asynccontextmanager combines both ideas.
# It turns an async generator function into an async context manager —
# everything before yield is __aenter__, everything after is __aexit__.
#
# In MCP, every server defines a lifespan this way.


# ─── WITHOUT @asynccontextmanager ────────────────────────────────────────────
# You already know this from 04_async_with.py — a class with __aenter__ / __aexit__.
# It works, but it is verbose for simple setup/teardown.

class DatabaseClass:
    async def __aenter__(self):
        print("[DB] connecting...")
        await asyncio.sleep(0.05)
        self.connected = True
        return self

    async def __aexit__(self, *args):
        print("[DB] closing...")
        await asyncio.sleep(0.02)
        self.connected = False


# ─── WITH @asynccontextmanager ────────────────────────────────────────────────
# Same behaviour, half the code.
# Rule: everything before yield = setup (__aenter__)
#       yield value               = what gets bound to `as db`
#       everything after yield    = teardown (__aexit__)

@asynccontextmanager
async def database() -> AsyncIterator[dict]:
    print("[DB] connecting...")
    await asyncio.sleep(0.05)
    conn = {"status": "open"}           # the object bound to `as db`
    yield conn                          # control passes to the `async with` body
    print("[DB] closing...")            # runs after the body exits
    await asyncio.sleep(0.02)
    conn["status"] = "closed"


async def compare():
    print("=== class-based ===")
    async with DatabaseClass() as db:
        print(f"  connected: {db.connected}")

    print("\n=== @asynccontextmanager ===")
    async with database() as db:
        print(f"  status: {db['status']}")


# ─── AsyncIterator[T] — the return type ──────────────────────────────────────
# An @asynccontextmanager function is an async generator under the hood.
# The correct type annotation for its return value is AsyncIterator[T]
# from collections.abc.
#
# You will see this on every lifespan function in the MCP course:
#
#   from collections.abc import AsyncIterator
#
#   @asynccontextmanager
#   async def app_lifespan(server) -> AsyncIterator[AppContext]:
#       ...
#       yield AppContext(db=db)
#       ...
#
# AsyncIterator[AppContext] tells the type checker:
# "this async generator yields AppContext objects".
# Without it the code still runs — it is a type hint, not runtime behaviour.


# ─── try / finally — guaranteed teardown even on errors ───────────────────────
# If the body of `async with` raises an exception, the code after yield
# still runs — but only if it is inside a finally block.

@asynccontextmanager
async def safe_connection(host: str) -> AsyncIterator[dict]:
    print(f"[{host}] connecting...")
    await asyncio.sleep(0.05)
    conn = {"host": host}
    try:
        yield conn                      # body runs here
    finally:
        print(f"[{host}] closing (always runs)")   # runs even if body raises
        await asyncio.sleep(0.02)


async def show_guaranteed_cleanup():
    print("\n=== cleanup runs even when body raises ===")
    try:
        async with safe_connection("localhost") as conn:
            print(f"  connected to {conn['host']}")
            raise ValueError("something went wrong")
    except ValueError as e:
        print(f"  caught: {e}")
    # [localhost] closing still printed above — teardown was guaranteed


# ─── MCP lifespan shape ───────────────────────────────────────────────────────
# This is the exact pattern every MCP server uses.
# The lifespan hook runs once at startup (before the server accepts any tool
# calls) and once at shutdown (after the last connection closes).
#
#   from collections.abc import AsyncIterator
#   from contextlib import asynccontextmanager
#   from dataclasses import dataclass
#   from mcp.server.fastmcp import FastMCP
#
#   @dataclass
#   class AppContext:
#       db: Database
#
#   @asynccontextmanager
#   async def app_lifespan(server: FastMCP) -> AsyncIterator[AppContext]:
#       db = await Database.connect()   # startup
#       try:
#           yield AppContext(db=db)     # context available to every tool via ctx
#       finally:
#           await db.disconnect()       # shutdown — always runs
#
#   mcp = FastMCP("my_server", lifespan=app_lifespan)


async def main():
    await compare()
    await show_guaranteed_cleanup()


if __name__ == "__main__":
    asyncio.run(main())

```

In [10]:
!python 04b_async_contextmanager.py

=== class-based ===
[DB] connecting...
  connected: True
[DB] closing...

=== @asynccontextmanager ===
[DB] connecting...
  status: open
[DB] closing...

=== cleanup runs even when body raises ===
[localhost] connecting...
  connected to localhost
[localhost] closing (always runs)
  caught: something went wrong


# Async For

`async for` is used when items arrive asynchronously — each one requires a wait before it's available, like reading a stream of responses from a server one at a time.

In MCP this appears when iterating paginated tool lists or reading streamed output from a model.

In [11]:
from IPython.display import display, Markdown

with open("05_async_for.py", encoding="utf-8") as f:
    code = f.read()

display(Markdown(f"```python\n{code}\n```"))

```python
import asyncio


# ─── WHAT IS async for? ─────────────────────────────────────────────────────
# A regular `for` loop calls __next__() synchronously — it expects each item to be available immediately.
# `async for` is used when each item requires waiting to arrive
# — like reading a stream of responses from a server one at a time.
#
# In MCP, async for appears when iterating paginated tool lists
# or reading streamed output from a model.


async def stream_tools():
    """Simulates an MCP server sending tool definitions one at a time."""
    tools = ["weather", "calculator", "file_reader"]
    for tool in tools:
        await asyncio.sleep(0.2)        # simulate each item arriving over the network
        yield {"name": tool, "status": "ok"}


async def main():
    print("Receiving tools from server:")
    async for tool in stream_tools():   # waits for each item before continuing
        print(f"  Got: {tool}")

    # Real MCP usage:
    #
    # async for tools_page in session.list_tools():
    #     for tool in tools_page:
    #         print(tool.name)


if __name__ == "__main__":
    asyncio.run(main())

```

In [12]:
!python 05_async_for.py

Receiving tools from server:
  Got: {'name': 'weather', 'status': 'ok'}
  Got: {'name': 'calculator', 'status': 'ok'}
  Got: {'name': 'file_reader', 'status': 'ok'}


# Full Pattern

All five async concepts combined into the shape of a real MCP client interaction:

- `async def` — every handler and helper is a coroutine
- `await` — used to call tools and wait for results
- `async with` — opens and closes the session safely
- `async for` — iterates the tool list from the server
- `asyncio.run()` — the single entry point that starts everything

In [13]:
from IPython.display import display, Markdown

with open("06_full_pattern.py", encoding="utf-8") as f:
    code = f.read()

display(Markdown(f"```python\n{code}\n```"))

```python
import asyncio


# ─── FULL PATTERN ───────────────────────────────────────────────────────────
# This file combines all five async concepts into the shape of a real
# MCP client interaction. Nothing here is simulated differently from
# what you'll write when building against an actual MCP server.
#
# Concepts used:
#   async def      — every handler and helper is a coroutine
#   await          — used to call tools and wait for results
#   asyncio.run()  — the single entry point that starts everything
#   async with     — opens and closes the session safely
#   async for      — iterates the tool list from the server


# ─── Simulated MCP server tool ──────────────────────────────────────────────

async def weather_tool(city: str) -> dict:
    await asyncio.sleep(0.2)            # simulate HTTP call to weather API
    return {"city": city, "temp": 72, "condition": "sunny"}


# ─── Simulated MCP session ──────────────────────────────────────────────────

class FakeSession:
    async def __aenter__(self):
        await asyncio.sleep(0.05)
        print("[Session] open")
        return self

    async def __aexit__(self, *args):
        await asyncio.sleep(0.05)
        print("[Session] closed")

    async def call_tool(self, name: str, args: dict):
        if name == "weather":
            return await weather_tool(args["city"])

    async def list_tools(self):
        for tool in ["weather", "calculator"]:
            await asyncio.sleep(0.05)
            yield tool


# ─── Main client logic ──────────────────────────────────────────────────────

async def main():
    async with FakeSession() as session:            # async with: safe open/close

        # async for: iterate tools as they arrive from the server
        print("Available tools:")
        async for tool_name in session.list_tools():
            print(f"  - {tool_name}")

        # asyncio.gather: call multiple tools concurrently
        cities = ["New York", "London", "Tokyo"]
        results = await asyncio.gather(
            *[session.call_tool("weather", {"city": c}) for c in cities]
        )

        print("\nWeather results:")
        for r in results:
            print(f"  {r['city']}: {r['temp']}F, {r['condition']}")


# ─── Entry point ────────────────────────────────────────────────────────────
# asyncio.run() creates the event loop, runs main(), then shuts it down.
# Every MCP server and client script ends this way.

if __name__ == "__main__":
    asyncio.run(main())

```

In [14]:
!python 06_full_pattern.py

[Session] open
Available tools:
  - weather
  - calculator

Weather results:
  New York: 72F, sunny
  London: 72F, sunny
  Tokyo: 72F, sunny
[Session] closed
